# 🧬 Agentic GWAS Cancer Risk Pipeline
### Cotiviti Intern Assessment — Layer 2: Agentic LLM Router
**Author:** Bhavya Sevak | M.S. Biomedical Informatics, Arizona State University | June 2026

---

## Architecture

```
[Patient Clinical Profile]
         │
         ▼
┌─────────────────────┐
│  Layer 1: XGBoost   │  ← GWAS/mQTL ML pipeline (AUC 0.81)
│  Cancer Risk Score  │    Flags High / Very High risk patients
└─────────────────────┘
         │  High Risk Detected?
         ▼
┌─────────────────────────────┐
│  Layer 2: Agentic LLM       │  ← Autonomous clinical reasoning
│  Router (Claude via API)    │    Triggers: Pre-Auth / Screening /
│                             │    Billing Optimization
└─────────────────────────────┘
         │
         ▼
  [Structured Clinical Action]
```

**Layer 1** runs the full GWAS ML pipeline and produces a PRS risk tier.  
**Layer 2** is an agentic LLM that receives the risk profile and autonomously decides which downstream clinical action to trigger — pre-authorization, screening referral, or billing optimization — and explains its reasoning in structured output.

> ⚠️ All data is synthetic. Not clinically validated.

In [1]:
# ── Install dependencies ──────────────────────────────────────────────────
!pip install numpy pandas scikit-learn xgboost anthropic --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 932.0/932.0 kB 9.6 MB/s eta 0:00:00


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import anthropic
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

np.random.seed(42)
print('✓ Libraries loaded')

✓ Libraries loaded


---
## Layer 1: GWAS/mQTL ML Pipeline

In [3]:
# ── Simulate GWAS Dataset ────────────────────────────────────────────────
N_SAMPLES, N_SNPS, N_CAUSAL = 1000, 5000, 50

mafs = np.random.uniform(0.05, 0.45, N_SNPS)
X_raw = np.array([
    np.random.choice([0,1,2], size=N_SAMPLES, p=[(1-m)**2, 2*m*(1-m), m**2])
    for m in mafs
]).T.astype(np.float32)

causal_idx = np.random.choice(N_SNPS, N_CAUSAL, replace=False)
effect_sizes = np.random.normal(0, 0.4, N_CAUSAL)

log_odds = -1.2
for k, idx in enumerate(causal_idx):
    mQTL = 1 + 0.3 * np.random.normal(size=N_SAMPLES)
    log_odds = log_odds + effect_sizes[k] * X_raw[:, idx] * mQTL

prob = 1 / (1 + np.exp(-log_odds))
y = (np.random.uniform(size=N_SAMPLES) < prob).astype(int)
snp_names = [f'rs_{i:06d}' for i in range(N_SNPS)]

print(f'✓ GWAS dataset: {N_SAMPLES} samples × {N_SNPS} SNPs | Cases: {y.sum()} ({y.mean():.1%})')

✓ GWAS dataset: 1000 samples × 5000 SNPs | Cases: 711 (71.1%)


In [4]:
# ── Feature Selection: Chi-squared → LASSO ───────────────────────────────
selector = SelectKBest(chi2, k=500)
X_filtered = selector.fit_transform(np.abs(X_raw), y)
selected_snps = np.array(snp_names)[selector.get_support()]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

lasso = LogisticRegressionCV(
    Cs=np.logspace(-3,1,20), cv=5, penalty='l1',
    solver='liblinear', max_iter=500, random_state=42
)
lasso.fit(X_scaled, y)
lasso_mask = lasso.coef_[0] != 0
X_lasso = X_scaled[:, lasso_mask]
lasso_snps = selected_snps[lasso_mask]

print(f'✓ Feature selection: 5,000 → 500 (chi2) → {lasso_mask.sum()} SNPs (LASSO)')

✓ Feature selection: 5,000 → 500 (chi2) → 361 SNPs (LASSO)


In [5]:
# ── Train XGBoost + Calibrate PRS ────────────────────────────────────────
clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(1-y.mean())/y.mean(),
    eval_metric='logloss', random_state=42, verbosity=0
)

cv_auc = cross_val_score(clf, X_lasso, y, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                          scoring='roc_auc', n_jobs=-1)
clf.fit(X_lasso, y)

cal_clf = CalibratedClassifierCV(clf, cv='prefit', method='isotonic')
cal_clf.fit(X_lasso, y)
prs = cal_clf.predict_proba(X_lasso)[:, 1]

risk_tiers = pd.cut(prs, bins=[0, 0.25, 0.50, 0.75, 1.0],
                     labels=['Low', 'Intermediate', 'High', 'Very High'])

print(f'✓ XGBoost CV AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}')
print(f'✓ PRS calibrated | Risk tiers assigned')

✓ XGBoost CV AUC: 0.861 ± 0.010
✓ PRS calibrated | Risk tiers assigned


In [6]:
# ── Build Patient Profiles (Layer 1 output → Layer 2 input) ──────────────
# Simulate 5 representative patients for the agentic demo
DEMO_PATIENTS = [
    {"id": "PT-001", "age": 58, "sex": "F", "ancestry": "Hispanic/Latino",
     "family_history": "Mother - colorectal cancer", "prior_colonoscopy": "None",
     "insurance": "Medicaid"},
    {"id": "PT-002", "age": 45, "sex": "M", "ancestry": "Non-Hispanic White",
     "family_history": "None", "prior_colonoscopy": "2019 - normal",
     "insurance": "Commercial PPO"},
    {"id": "PT-003", "age": 63, "sex": "M", "ancestry": "Black/African American",
     "family_history": "Father - prostate cancer", "prior_colonoscopy": "None",
     "insurance": "Medicare"},
    {"id": "PT-004", "age": 39, "sex": "F", "ancestry": "South Asian",
     "family_history": "None", "prior_colonoscopy": "None",
     "insurance": "Commercial HMO"},
    {"id": "PT-005", "age": 71, "sex": "F", "ancestry": "Non-Hispanic White",
     "family_history": "Sister - breast cancer, Mother - colorectal cancer",
     "prior_colonoscopy": "2015 - polyp removed", "insurance": "Medicare"},
]

# Assign synthetic PRS scores (from pipeline distribution)
DEMO_PRS = [0.81, 0.22, 0.76, 0.31, 0.91]
DEMO_TIERS = ['Very High', 'Low', 'High', 'Intermediate', 'Very High']
TOP_SNPS = lasso_snps[:5].tolist()  # top 5 flagged SNPs

for i, p in enumerate(DEMO_PATIENTS):
    p['prs_score'] = DEMO_PRS[i]
    p['risk_tier'] = DEMO_TIERS[i]
    p['flagged_snps'] = TOP_SNPS

print('✓ Patient profiles constructed:')
for p in DEMO_PATIENTS:
    print(f"  {p['id']} | PRS: {p['prs_score']} | Tier: {p['risk_tier']} | Age: {p['age']} | {p['ancestry']}")

✓ Patient profiles constructed:
  PT-001 | PRS: 0.81 | Tier: Very High | Age: 58 | Hispanic/Latino
  PT-002 | PRS: 0.22 | Tier: Low | Age: 45 | Non-Hispanic White
  PT-003 | PRS: 0.76 | Tier: High | Age: 63 | Black/African American
  PT-004 | PRS: 0.31 | Tier: Intermediate | Age: 39 | South Asian
  PT-005 | PRS: 0.91 | Tier: Very High | Age: 71 | Non-Hispanic White


---
## Layer 2: Agentic LLM Router

The agentic layer receives each patient's risk profile from Layer 1 and autonomously:
- **Reasons** about the clinical context
- **Selects** the appropriate downstream action
- **Outputs** structured JSON for downstream systems (EHR, payer platform, billing)

Available actions:
| Action | Trigger Condition |
|--------|------------------|
| `TRIGGER_PRE_AUTHORIZATION` | High/Very High risk + no recent screening |
| `APPROVE_EARLY_SCREENING` | High/Very High risk + age < 45 or specific ancestry |
| `OPTIMIZE_BILLING_CODE` | Any risk tier — maps to correct CPT/ICD-10 codes |
| `ROUTINE_MONITORING` | Low/Intermediate risk |
| `ESCALATE_TO_ONCOLOGY` | Very High risk + multiple family history flags |

In [12]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.6 MB/s eta 0:00:00


In [16]:
# ── Groq API Setup ────────────────────────────────────────────────────────
from groq import Groq

GROQ_API_KEY = "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"

client = Groq(api_key=GROQ_API_KEY)
print('✓ Groq client initialized')

✓ Groq client initialized


In [19]:
def run_agentic_router(patient: dict) -> dict:
    """Send patient profile to Groq LLM agent and return structured clinical action."""

    user_message = f"""Patient profile from GWAS ML pipeline:

Patient ID: {patient['id']}
Age: {patient['age']} | Sex: {patient['sex']} | Ancestry: {patient['ancestry']}
Family History: {patient['family_history']}
Prior Colonoscopy: {patient['prior_colonoscopy']}
Insurance: {patient['insurance']}

ML Pipeline Output:
- Polygenic Risk Score (PRS): {patient['prs_score']} (scale 0-1)
- Risk Tier: {patient['risk_tier']}
- Top flagged SNPs: {', '.join(patient['flagged_snps'])}
- Model: XGBoost trained on GWAS/mQTL data (CV AUC 0.81)

Determine the appropriate clinical action and output JSON."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ],
        temperature=0.2,
        max_tokens=1000,
    )

    raw = response.choices[0].message.content.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
    return json.loads(raw.strip())

print('✓ Agentic router function defined')

✓ Agentic router function defined


In [20]:
# ── Run Agentic Pipeline on All Demo Patients ─────────────────────────────
print('=' * 65)
print('  LAYER 2: AGENTIC LLM ROUTER — PROCESSING PATIENTS')
print('=' * 65)

results = []

for patient in DEMO_PATIENTS:
    print(f"\n🔄 Processing {patient['id']} (PRS: {patient['prs_score']} | {patient['risk_tier']})...")
    try:
        action = run_agentic_router(patient)
        results.append(action)

        print(f"  ✅ Primary Action : {action['primary_action']}")
        if action.get('secondary_action'):
            print(f"  ✅ Secondary      : {action['secondary_action']}")
        print(f"  🚨 Urgency        : {action['urgency']}")
        print(f"  🧠 Reasoning      : {action['clinical_reasoning']}")
        print(f"  💰 CPT Codes      : {', '.join(action.get('recommended_cpt_codes', []))}")
        print(f"  🏥 ICD-10         : {', '.join(action.get('recommended_icd10', []))}")
        if action.get('equity_flag'):
            print(f"  ⚖️  Equity Note   : {action['equity_flag']}")

    except Exception as e:
        print(f"  ❌ Error: {e}")
        results.append({"patient_id": patient['id'], "error": str(e)})

print('\n' + '=' * 65)
print('  PIPELINE COMPLETE')
print('=' * 65)

  LAYER 2: AGENTIC LLM ROUTER — PROCESSING PATIENTS

🔄 Processing PT-001 (PRS: 0.81 | Very High)...
  ✅ Primary Action : TRIGGER_PRE_AUTHORIZATION
  ✅ Secondary      : APPROVE_EARLY_SCREENING
  🚨 Urgency        : URGENT
  🧠 Reasoning      : The patient's very high Polygenic Risk Score (PRS) of 0.81, combined with a family history of colorectal cancer, indicates an elevated risk of developing colorectal cancer. Given the patient's age and lack of prior colonoscopy, early screening is warranted. Pre-authorization for colonoscopy and genetic counseling is necessary to ensure timely and appropriate care.
  💰 CPT Codes      : 82270, 81292
  🏥 ICD-10         : Z80.0
  ⚖️  Equity Note   : PRS calibration considerations for Hispanic/Latino ancestry may be necessary, as genetic risk scores can vary across populations.

🔄 Processing PT-002 (PRS: 0.22 | Low)...
  ✅ Primary Action : ROUTINE_MONITORING
  🚨 Urgency        : ROUTINE
  🧠 Reasoning      : The patient's Polygenic Risk Score (PRS) of 0.2

In [21]:
# ── Summary Table ─────────────────────────────────────────────────────────
print('\n📊 AGENTIC PIPELINE SUMMARY\n')
print(f'{"Patient":<10} {"PRS":<6} {"Tier":<15} {"Primary Action":<30} {"Urgency":<10}')
print('-' * 75)

for patient, result in zip(DEMO_PATIENTS, results):
    action = result.get('primary_action', 'ERROR')
    urgency = result.get('urgency', '-')
    print(f"{patient['id']:<10} {patient['prs_score']:<6} {patient['risk_tier']:<15} {action:<30} {urgency:<10}")

print('\n✅ Layer 1 (GWAS ML) + Layer 2 (Agentic LLM) pipeline complete.')
print('   This output is ready to feed into EHR CDS hooks, payer dashboards,')
print('   or pre-authorization workflows via FHIR API.')


📊 AGENTIC PIPELINE SUMMARY

Patient    PRS    Tier            Primary Action                 Urgency   
---------------------------------------------------------------------------
PT-001     0.81   Very High       TRIGGER_PRE_AUTHORIZATION      URGENT    
PT-002     0.22   Low             ROUTINE_MONITORING             ROUTINE   
PT-003     0.76   High            APPROVE_EARLY_SCREENING        URGENT    
PT-004     0.31   Intermediate    APPROVE_EARLY_SCREENING        ROUTINE   
PT-005     0.91   Very High       APPROVE_EARLY_SCREENING        URGENT    

✅ Layer 1 (GWAS ML) + Layer 2 (Agentic LLM) pipeline complete.
   This output is ready to feed into EHR CDS hooks, payer dashboards,
   or pre-authorization workflows via FHIR API.


---
## What This Demonstrates for Cotiviti

| Layer | Technology | Cotiviti Value |
|-------|-----------|----------------|
| Layer 1: GWAS ML | XGBoost + LASSO + PRS calibration | Population health risk stratification |
| Layer 2: Agentic LLM | Claude via Anthropic API | Autonomous TPO decision routing |
| Output | Structured JSON | Plug into FHIR CDS hooks, payer platforms, billing systems |

**Equity awareness** — the agent flags ancestry-specific PRS calibration concerns (Hispanic/Latino, South Asian cohorts where training data may be sparse), directly addressing known health equity gaps in genomic AI.

**Treatment, Payment & Operations (TPO)** — the pipeline autonomously handles all three: clinical treatment decisions, pre-authorization/billing code optimization, and operational escalation routing.

---
*Synthetic data only. Not clinically validated. Built for Cotiviti Intern Assessment.*